# Notebook de EDA + Limpeza de dados Clientes

## Configuração de Ambiente


### Bibliotecas python

In [35]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

### Caminho base do projeto

In [36]:
BASE_PATH = Path().resolve()

while BASE_PATH.name != "lh-nautical-data-project":
    BASE_PATH = BASE_PATH.parent

print(f"BASE PATH: {BASE_PATH}")

BASE PATH: /Users/richardgomes/lh-nautical-data-project


### Caminhos para os dados raw e staging

In [37]:
DATA_PATH = BASE_PATH / "data"
RAW_PATH = DATA_PATH / "raw"
STAGING_PATH = DATA_PATH / "staging"

In [38]:
print(f"RAW PATH: {RAW_PATH}")
print(f"STAGING PATH: {STAGING_PATH}")

RAW PATH: /Users/richardgomes/lh-nautical-data-project/data/raw
STAGING PATH: /Users/richardgomes/lh-nautical-data-project/data/staging


#### Leitura e carregamento dos dados dos clientes vindos do CRM (clientes_crm.json)

In [39]:
with open(RAW_PATH / "clientes_crm.json", "r", encoding="utf-8") as f:
    clientes_crm_json = json.load(f)

In [40]:
df_clientes = pd.json_normalize(clientes_crm_json)

In [41]:
df_clientes.shape

(49, 4)

In [42]:
df_clientes.head(10)

,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com
5,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO",6,coelho.pinheiro.peixoto.antônia.cavalcanti@aol...
6,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo,7,torres.barros.rocha.bianca.siqueira#aol.com
7,Luiz Alves Pimentel,SE - Aracaju,8,pimentel.alves.luiz#outlook.com
8,Lucas Guedes Cunha Lopes,PB - João Pessoa,9,lucas.lopes.guedes.cunha#tutanota.com
9,Débora Paiva,Santarém / PA,10,paiva.débora#gmx.com


Notei problemas com os dados em duas colunas (location e email)
Há presença de # no lugar de @ em alguns emails 
E location não segue um padrão.

In [43]:
df_clientes[df_clientes['email'].str.contains('#', na=False)]

,full_name,location,code,email
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com
6,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo,7,torres.barros.rocha.bianca.siqueira#aol.com
7,Luiz Alves Pimentel,SE - Aracaju,8,pimentel.alves.luiz#outlook.com
8,Lucas Guedes Cunha Lopes,PB - João Pessoa,9,lucas.lopes.guedes.cunha#tutanota.com
9,Débora Paiva,Santarém / PA,10,paiva.débora#gmx.com
11,Rafael Pereira Barros,"TO , Fortaleza do Tabocão",12,rafael.pereira.barros#zoho.com
12,Carlos Guimarães Martins,PA / Santarém,13,martins.guimarães.carlos#hotmail.com
14,Carla Lopes Alves Pacheco Rocha,"Fortaleza do Tabocão,TO",15,lopes.alves.pacheco.rocha.carla#yahoo.com


In [44]:
df_clientes['email'].str.contains('#', na=False).sum()

np.int64(30)

61.22% são emails "invalidos" com # e não @. Isso indica algum tipo de falha frave no CRM

In [45]:
total = len(df_clientes)
invalidos = df_clientes['email'].str.contains('#', na=False).sum()

print(f"{invalidos} de {total} emails inválidos ({invalidos/total:.2%})")

30 de 49 emails inválidos (61.22%)


Como existe um padrão claro de erros, podemos corrigir.  

In [46]:
df_clientes['email'] = df_clientes['email'].str.replace('#', '@', regex=False)

In [47]:
df_clientes['email'].str.contains('#', na=False).sum()

np.int64(0)

In [48]:
df_clientes.head(10)

,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro@gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira@gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas@icloud.com
5,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO",6,coelho.pinheiro.peixoto.antônia.cavalcanti@aol...
6,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo,7,torres.barros.rocha.bianca.siqueira@aol.com
7,Luiz Alves Pimentel,SE - Aracaju,8,pimentel.alves.luiz@outlook.com
8,Lucas Guedes Cunha Lopes,PB - João Pessoa,9,lucas.lopes.guedes.cunha@tutanota.com
9,Débora Paiva,Santarém / PA,10,paiva.débora@gmx.com


Agora, a padronização das localidades. 

In [49]:
df_clientes['location_tratada'] = (
    df_clientes['location']
    .str.replace(r'\s+', ' ', regex=True)   
    .str.replace(r'[/\-]', ',', regex=True) 
    .str.replace(r'\s*,\s*', ',', regex=True) 
    .str.strip()
)

In [50]:
df_clientes[['location', 'location_tratada']].head(10)

,location,location_tratada
0,"Aratu (Candeias) , BA","Aratu (Candeias),BA"
1,"PE , Recife","PE,Recife"
2,"Rio Grande,RS","Rio Grande,RS"
3,"AC , Rio Branco","AC,Rio Branco"
4,PA - Santarém Novo,"PA,Santarém Novo"
5,"Fortaleza do Tabocão , TO","Fortaleza do Tabocão,TO"
6,PB/Cabedelo,"PB,Cabedelo"
7,SE - Aracaju,"SE,Aracaju"
8,PB - João Pessoa,"PB,João Pessoa"
9,Santarém / PA,"Santarém,PA"


In [51]:
def separar_cidade_estado(valor):
    if pd.isna(valor):
        return pd.Series([None, None])

    partes = valor.split(',')

    if len(partes) != 2:
        return pd.Series([None, None])

    p1, p2 = partes[0].strip(), partes[1].strip()

    if len(p1) == 2:
        estado = p1
        cidade = p2
    
    elif len(p2) == 2:
        cidade = p1
        estado = p2
    else:
        return pd.Series([None, None])



    return pd.Series([cidade, estado])


df_clientes[['cidade', 'estado']] = df_clientes['location_tratada'].apply(separar_cidade_estado)

In [52]:
df_clientes['cidade'] = df_clientes['cidade'].str.replace(r'\s*\(.*?\)', '', regex=True)

In [53]:
df_clientes[['location_tratada', 'cidade', 'estado']].head(10)

,location_tratada,cidade,estado
0,"Aratu (Candeias),BA",Aratu,BA
1,"PE,Recife",Recife,PE
2,"Rio Grande,RS",Rio Grande,RS
3,"AC,Rio Branco",Rio Branco,AC
4,"PA,Santarém Novo",Santarém Novo,PA
5,"Fortaleza do Tabocão,TO",Fortaleza do Tabocão,TO
6,"PB,Cabedelo",Cabedelo,PB
7,"SE,Aracaju",Aracaju,SE
8,"PB,João Pessoa",João Pessoa,PB
9,"Santarém,PA",Santarém,PA


In [54]:
df_clientes[['cidade', 'estado']].isnull().sum()

cidade    0
estado    0
dtype: int64

In [55]:
df_clientes = df_clientes.drop(columns=['location', 'location_tratada'])

In [56]:
df_clientes = df_clientes.rename(columns={
    'code': 'id_cliente',
    'full_name': 'nome_cliente'
})

In [57]:
df_clientes = df_clientes[
    [
        'id_cliente',
        'nome_cliente',
        'email',
        'cidade',
        'estado'
    ]
]

In [58]:
df_clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_cliente    49 non-null     int64
 1   nome_cliente  49 non-null     str  
 2   email         49 non-null     str  
 3   cidade        49 non-null     str  
 4   estado        49 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.0 KB


In [59]:
df_clientes = df_clientes.sort_values(by='id_cliente').reset_index(drop=True)

In [60]:
df_clientes.duplicated(subset='id_cliente').sum()

np.int64(0)

In [61]:
df_clientes.head()

,id_cliente,nome_cliente,email,cidade,estado
0,1,Femininos Oliveira Antunes,femininos.oliveira.antunes@icloud.com,Aratu,BA
1,2,Fernanda Azevedo Soares Nunes Vieira,nunes.fernanda.soares.azevedo.vieira@outlook.com,Recife,PE
2,3,Daniel Farias Ribeiro Teixeira,farias.teixeira.daniel.ribeiro@gmail.com,Rio Grande,RS
3,4,Thiago Moreira,thiago.moreira@gmail.com,Rio Branco,AC
4,5,Pedro Freitas,pedro.freitas@icloud.com,Santarém Novo,PA


Conferindo as colunas

In [62]:
df_clientes.columns

Index(['id_cliente', 'nome_cliente', 'email', 'cidade', 'estado'], dtype='str')

In [63]:
df_clientes['id_cliente'] = df_clientes['id_cliente'].astype(int)
df_clientes['nome_cliente'] = df_clientes['nome_cliente'].astype(str)
df_clientes['email'] = df_clientes['email'].astype(str)
df_clientes['cidade'] = df_clientes['cidade'].astype(str)
df_clientes['estado'] = df_clientes['estado'].astype(str)

In [64]:
df_clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_cliente    49 non-null     int64
 1   nome_cliente  49 non-null     str  
 2   email         49 non-null     str  
 3   cidade        49 non-null     str  
 4   estado        49 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.0 KB


Padronização das strings

In [65]:
df_clientes['cidade'] = df_clientes['cidade'].str.title().str.strip()
df_clientes['estado'] = df_clientes['estado'].str.upper().str.strip()
df_clientes['nome_cliente'] = df_clientes['nome_cliente'].str.strip()
df_clientes['email'] = df_clientes['email'].str.lower().str.strip()

Validação

In [66]:
print("Duplicados:", df_clientes.duplicated(subset='id_cliente').sum())
print("Nulos:\n", df_clientes.isnull().sum())
print("Shape:", df_clientes.shape)

Duplicados: 0
Nulos:
 id_cliente      0
nome_cliente    0
email           0
cidade          0
estado          0
dtype: int64
Shape: (49, 5)


Exportação para a camada Staging

In [67]:
df_clientes.to_csv(STAGING_PATH / "stg_clientes.csv", index=False)
print("Arquivo salvo em:", STAGING_PATH / "stg_clientes.csv")

Arquivo salvo em: /Users/richardgomes/lh-nautical-data-project/data/staging/stg_clientes.csv
